In [1]:
import joblib
import numpy as np
import pandas as pd
import torch
from bolero import init
from bolero.tl.model.borzoi.train import BorzoiLoRATrainer
import anndata
init(num_cpus=32, verbose=True)

Enabled torch cudnn
Disabled ray lineage reconstruction and task retries


2024-10-14 16:28:13,310	INFO worker.py:1762 -- Started a local Ray instance. View the dashboard at 127.0.0.1:8265 


In [2]:
name = "test-borzoi-trainer-with-online-loader"
# use_kv_bottleneck = 1
use_kv_bottleneck = 0
use_ema = 0
context_out = 1
n_pseudobulks = 2
fold_id = 0
lora_preset = "all_conditional"

# kv_bottleneck = "local"
kv_bottleneck = None
n_cycles = 1
cov_soft_clamp = None
loss_total_weight = 0.2

In [3]:
# warmup_steps = 10000
# lr = 5e-5
# train_batches = 5000
# val_batches = 1000

# for quick test
warmup_steps = 100
lr = 1e-3
train_batches = 2000
val_batches = 500


In [4]:
data_dir = '/large_storage/zhoulab/tlgallent/data/Li2023Science/'
fold_id = 0


In [5]:
adata = anndata.read_h5ad('/large_storage/zhoulab/tlgallent/data/cell_29000_rna_raw_gencode_adata_with_embeddings.h5ad')
scvi_embedding = adata.obsm['X_scVI']
# Create a DataFrame with embeddings and cell types
df = pd.DataFrame(scvi_embedding, index=adata.obs.index)
df['cell_type'] = adata.obs['MajorType']
# Group by cell type
grouped = df.groupby('cell_type').mean()
# Get embedding
recalculated_embedding = grouped.to_numpy()


leg_map = {item: recalculated_embedding[index] for index, item in enumerate(grouped.index.to_list())}
leg_map['ASC'].shape

/tmp/ipykernel_1854332/42406001.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = df.groupby('cell_type').mean()


(50,)

In [6]:
checkpoint_path = f"/large_storage/zhoulab/hanliu/240729-WMBRNAModel/05.borzoi/borzoi_pretrain/torch_checkpoints/borzoi.human.f{fold_id}.pt"
dataset_path = (
    f"{data_dir}"
)
vq_records_path = "/home/hanliu/data/240729-WMBRNAModel/02.scVI/scVI-VQ/ALL_vq/vq_records.target_frags_8000000.joblib"
emb_input_features = 50
use_ema = False if use_ema == 0 else True
context_out = False if context_out == 0 else True

In [7]:
config = {
    # file path and wandb
    "output_dir": "model",
    "savename": name,
    "wandb_project": "borzoi_finetuning",
    "wandb_job_type": "lora",
    "wandb_name": f"{name}-fold-{fold_id}",
    # train
    "start_early_stop_after_epoch": 15,
    "max_epochs": 30,
    "patience": 10,
    "use_amp": True,
    "use_ema": use_ema,
    "train_batches": train_batches,
    "val_batches": val_batches,
    "batch_size": 2,
    "accumulate_grad": 64,
    "lr": lr,
    "warmup_steps": warmup_steps,
    "weight_decay": 1e-8,
    "global_clipnorm": 0.1,
    # dataset
    # "dataset_path": dataset_path,
    "fold_split_id": 0,
    "shuffle_rows": 300,
    "dna_window": 524288,
    "pos_resolution": 32,
    "reverse_complement": True,
    "max_jitter": 3,
    # "n_pseudobulks": n_pseudobulks,
    # LoRA Model
    "base_checkpoint_path": checkpoint_path,
    "emb_input_features": emb_input_features,
    "out_channels": 1,
    "hidden_dim": 256,
    "hidden_layers": 1,
    "output_layer_groups": 4,
    "lora_dropout": 0.01,
    "loss_total_weight": loss_total_weight,
    "lora_preset": lora_preset,
    "context_output": context_out,
    # "lora_norm": "layer",
    # base model dropout:
    "transformer_attn_dropout": 0.0,
    "transformer_pos_dropout": 0.0,
    "transformer_ff_dropout": 0.0,
    "final_conv_dropout": 0.0,
    "final_output_dropout": 0.01,
    # pseudobulk and embedding
    # "vq_records": vq_records_path,
    # "target_cov": 8e6,
    # "use_vq_emb": not use_kv_bottleneck,
    "kv_bottleneck": kv_bottleneck,
    "num_memories": 256,
    "dim_memory": 20,
    "num_memory_codebooks": 2,
    "additional_embs": 1,
    "n_cycles": n_cycles,
    # "cov_soft_clamp": cov_soft_clamp,
    #Online Loader reqs
    'bigwig_paths': [f"{data_dir}bigwig/ASC.bw", f"{data_dir}bigwig/Vip.bw"],
    'bed': None,
    'embeddings_path': '/large_storage/zhoulab/tlgallent/data/cell_29000_rna_raw_gencode_adata_with_embeddings.h5ad',
    'lora': True,
    # 'use_borzoi_regions': True,
}

config = BorzoiLoRATrainer.make_config(**config)

In [8]:
trainer = BorzoiLoRATrainer(config)

Create BorzoiDatasetOnline with config: {'bigwig_paths': ['/large_storage/zhoulab/tlgallent/data/Li2023Science/bigwig/ASC.bw', '/large_storage/zhoulab/tlgallent/data/Li2023Science/bigwig/Vip.bw'], 'bed': None, 'embeddings_path': '/large_storage/zhoulab/tlgallent/data/cell_29000_rna_raw_gencode_adata_with_embeddings.h5ad', 'lora': True, 'genome': 'hg38', 'batch_size': 2, 'dna_window': 524288, 'pos_resolution': 32, 'reverse_complement': True, 'max_jitter': 3, 'clamp_sqrt_threshold': None, 'shuffle_files': True, 'use_borzoi_regions': True}


/home/tlgallent/projects/software/bolero/src/bolero/tl/model/borzoi/dataset.py:492: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = df.groupby('cell_type').mean()


In [9]:
trainer.train(skip_output_adjust=True)

Training output layer only
Setting up model from config
Loading base model weights from: /large_storage/zhoulab/hanliu/240729-WMBRNAModel/05.borzoi/borzoi_pretrain/torch_checkpoints/borzoi.human.f0.pt
Total model parameters 195429893, trainable parameters 24157925
EMA is not used.
Training LoRA model
Layer (type (var_name))                                           Input Shape               Output Shape              Param #
BorzoiLoRA (BorzoiLoRA)                                           --                        [1, 1, 16352]             --
├─ConvDna (conv_dna)                                              [1, 4, 524288]            [1, 512, 262144]          --
│    └─ConditionalLoRAConv (conv_layer)                           [1, 4, 524288]            [1, 512, 524288]          31,232
│    │    └─EmbeddingMLP (lora_A_module)                          --                        [1, 45, 60]               255,373
│    │    └─EmbeddingMLP (lora_B_module)                          --           

/home/tlgallent/miniconda/envs/bolero_dev/lib/python3.10/site-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
2024-10-14 16:28:57,762	INFO streaming_executor.py:108 -- Starting execution of Dataset. Full logs are in /tmp/ray/session_2024-10-14_16-28-10_642480_1854332/logs/ray-data
2024-10-14 16:28:57,762	INFO streaming_executor.py:109 -- Execution plan of Dataset: InputDataBuffer[Input] -> AllToAllOperator[Repartition]


- Repartition 1:   0%|                                                                                        …

Split Repartition 2:   0%|                                                                                    …

Running 0:   0%|                                                                                              …

...

Running 0:   0%|                                                                                              …

 - (Validation) 0 [49/500] Mean Loss: 0.347; Mean Track Corr: 0.759; 
 - (Validation) 0 [99/500] Mean Loss: 0.352; Mean Track Corr: 0.764; 
 - (Validation) 0 [149/500] Mean Loss: 0.348; Mean Track Corr: 0.761; 
 - (Validation) 0 [199/500] Mean Loss: 0.340; Mean Track Corr: 0.759; 
 - (Validation) 0 [249/500] Mean Loss: 0.343; Mean Track Corr: 0.758; 
 - (Validation) 0 [299/500] Mean Loss: 0.334; Mean Track Corr: 0.759; 
 - (Validation) 0 [349/500] Mean Loss: 0.332; Mean Track Corr: 0.759; 
 - (Validation) 0 [399/500] Mean Loss: 0.331; Mean Track Corr: 0.758; 
 - (Validation) 0 [449/500] Mean Loss: 0.334; Mean Track Corr: 0.758; 
 - (Validation) 0 [499/500] Mean Loss: 0.335; Mean Track Corr: 0.759; 


/home/tlgallent/miniconda/envs/bolero_dev/lib/python3.10/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/tlgallent/miniconda/envs/bolero_dev/lib/python3.10/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


 - (Training)   1 Loss: 0.320; Mean Corr. : 0.532; Learning rate 0.001.
 - (Validation) 1 Loss: 0.335; Mean Corr. : 0.759.
Previous best metric: -1.000, Metric at epoch 1: 0.759; Early stopping counter: 0
Saving best checkpoint...


Error: You must call wandb.init() before wandb.log()

In [ ]:
a, b = torch.randn(3,1, 5, requires_grad=True), torch.randn(3, 5)
print(a.shape)
print(b.squeeze(1).shape)


In [ ]:
torch.nn.MSELoss(a,b)